In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from imblearn.under_sampling import RandomUnderSampler
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from global_vars import chargement_df

In [ ]:
df = pd.read_csv("../data/raw/accidents_2019_2023.csv")

In [ ]:
# changement ordre de gravité pour avoir un ordre logique 
df['grav'] = df['grav'].replace({2: 42})
df['grav'] = df['grav'].replace({4: 2})
df['grav'] = df['grav'].replace({42: 4})

In [ ]:
# modification de lat et long pour avoir le bon style

df['lat'] = df['lat'].str.replace(',', '.')
df['long'] = df['long'].str.replace(',', '.')

df['lat'] = df['lat'].astype(float)
df['long'] = df['long'].astype(float)

In [ ]:
# création d'une variable age au moment de l'accident
df['age'] = df['an']-df['an_nais']
df.drop(['an_nais'],axis=1,inplace = True)

# remplace les valeurs abérantes > 105 ans

df['age'] = df['age'].replace({106: 6})
df['age'] = df['age'].replace({108: 8})
df['age'] = df['age'].replace({109: 9})
df['age'] = df['age'].replace({110: 10})
df['age'] = df['age'].replace({118: 18})
df['age'] = df['age'].replace({119: 19})
df['age'] = df['age'].replace({120: 20})

# fonction pour ramplacer des valeurs manquantes dans age par une valeur aléatoire parmis 
# les valeurs observées en respectant la distribution observée

def replace_nan_with_random(df, column_name):
    observed_values = df[column_name].dropna()
    random_values = np.random.choice(observed_values, size=df[column_name].isna().sum())
    df.loc[df[column_name].isna(), column_name] = random_values
    return df

# Remplacer les NaN dans 'age'
df = replace_nan_with_random(df, 'age')



In [ ]:
# suppression des valeurs manquantes pour la variable cible

df = df[accidents_copy['grav'] != -1]

In [ ]:
# Définition de la fonction pour homogénéiser le format de l'heure
def homogenize_hour_format(row):
    # Convertir l'heure en chaîne de caractères
    hour_str = str(row['hrmn'])
    
    # Si l'année est comprise entre 2005 et 2018, ajuster le format de l'heure
    if row['an'] < 2019:
        # Extraire les deux derniers chiffres pour les minutes
        minutes = hour_str[-2:].zfill(2)
        
        # extrait les autre pour les heures
        hour = hour_str[:-2].zfill(2)
        
        #concatene avec ':' pour obtenir le format 'HH:MM'
        return f'{hour}:{minutes}'
    
    # Si l'année est 2019 ou plus, le format est déjà 'HH:MM'
    return hour_str

# Appliquer la fonction à la colonne 'hrmn'
df['hrmn'] = df.apply(homogenize_hour_format, axis=1)

In [ ]:
# remplacement des valeurs manquante de sexe suivant leur proportion dans la catégorie
# grav = 1 (toutes les valeurs manquantes sont dans cette catégorie)

proportion = [1] * 71 + [2] * 29

n_manquants = (df['sexe'] == -1).sum()
valeurs_remplacement = np.random.choice(proportion, size=n_manquants, replace=True)
df.loc[df['sexe'] == -1, 'sexe'] = valeurs_remplacement

In [ ]:
# remplacement des valeurs abérrantes vma
valeurs_50 = [500, 55, 520, 501, 502, 5]
valeurs_90 = [900, 901, 9]
valeurs_80 = [800, 180, 8]
valeurs_70 = [7, 75, 700, 770]
valeurs_60 = [560, 600]
valeurs_30 = [3, 300, 31, 35]
valeurs_40 = [140, 4, 42]

df['vma'] = df['vma'].replace(valeurs_50, 50)
df['vma'] = df['vma'].replace(valeurs_70, 70)
df['vma'] = df['vma'].replace(valeurs_80, 80)
df['vma'] = df['vma'].replace(valeurs_90, 90)
df['vma'] = df['vma'].replace(valeurs_60, 60)
df['vma'] = df['vma'].replace(valeurs_30, 30)
df['vma'] = df['vma'].replace(valeurs_40, 40)

# remplacement des -1 en fonction de la catégorie de route

df.loc[(df['vma'] == -1) &
                   (df['catr'] == 1), 'vma'] = 130

df.loc[(df['vma'] == -1) &
                   (df['catr'] == 2), 'vma'] = 80

df.loc[(df['vma'] == -1) &
                   (df['catr'] == 6), 'vma'] = 30

df['vma'] = df['vma'].replace({-1: 50})

df.loc[df['vma'] < 30, 'vma'] = 30


# regroupement des catégories rares avec une catégorie proche

df['vma'] = df['vma'].replace({40: 50})
df['vma'] = df['vma'].replace({45: 50})
df['vma'] = df['vma'].replace({60: 70})
df['vma'] = df['vma'].replace({65: 70})
df['vma'] = df['vma'].replace({100: 110})
df['vma'] = df['vma'].replace({120: 130})
                                                      
# les vitesses à 130 hors autoroute ramenées à 30

df.loc[(df['vma'] == 130) &
                   (df['catr'] != 1), 'vma'] = 30

#les vitesses >70 sur voies cummunales ramenées à 70
df.loc[(df['vma'] > 70) &
                   (df['catr'] == 4), 'vma'] = 70



In [ ]:
# remplacement des valeurs manquantes de lum en fonction des heures de la journée

# Remplacer les -1 par 1 entre 9h et 17h
df.loc[(df['lum'] == -1) &
                   (df['hrmn'] > '09:00') &
                   (df['hrmn'] < '17:00'), 'lum'] = 1

# Remplacer les -1 par 3 avant 7h ou après 20h
df.loc[(df['lum'] == -1) &
                   ((df['hrmn'] < '07:00') |
                    (df['hrmn'] > '20:00')), 'lum'] = 3

# Remplacer les -1 restants par 2
df['lum'] = df['lum'].replace({-1: 2})

In [ ]:
# remplacement des NAs par le mode de la colonne pour toutes les variables concernées
imputer = SimpleImputer(strategy='most_frequent')
cols_to_impute = [''] # A remplir avec les colonnes concernées
df[cols_to_impute] = imputer.fit_transform(df[cols_to_impute])

In [ ]:
###############################
# #gestion des variables secu1, secu2, secu3 nico le 20/06/2025
# #############################
# 
# Étape 1 — Nettoyer les données (remplacer -1 par NaN)
df[['secu1', 'secu2']] = df[['secu1', 'secu2']].replace(-1, np.nan)

# Étape 2 — Fonction pour calculer la distribution conditionnelle par colonne
def get_proba_by_col(df, colname, group_cols=['catu', 'catv']):
    temp = df[df[colname].notna()].copy()
    temp['profil'] = temp[group_cols].astype(str).agg('_'.join, axis=1)
    return (
        temp.groupby('profil')[colname]
        .value_counts(normalize=True)
        .unstack(fill_value=0)
    )

# Étape 3 — Calcul des distributions conditionnelles séparées
proba_secu1 = get_proba_by_col(df, 'secu1')
proba_secu2 = get_proba_by_col(df, 'secu2')


# Étape 4 — Fonction d’imputation basée sur les distributions
def imputer_secu(df, col, proba_df, profil_cols):
    def impute(row):
        if pd.notna(row[col]):
            return row[col]
        profil_key = "_".join([str(row[c]) for c in profil_cols])
        if profil_key not in proba_df.index:
            return np.nan  # fallback possible ici
        p = proba_df.loc[profil_key]
        return np.random.choice(p.index, p=p.values)
    return df.apply(impute, axis=1)

# Étape 5 — Imputation indépendante des 3 colonnes
df['secu1_corr'] = imputer_secu(df, 'secu1', proba_secu1, ['catu', 'catv'])
df['secu2_corr'] = imputer_secu(df, 'secu2', proba_secu2, ['catu', 'catv'])


# Étape 6 — Regrouper les équipements corrigés dans une seule liste
def regrouper_equipements(row):
    return [x for x in [row['secu1_corr'], row['secu2_corr']] if pd.notna(x)]

df['equipements'] = df.apply(regrouper_equipements, axis=1)

#encodage de la liste 'equipement'regrouper_equipements
from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()
equip_ohe = pd.DataFrame(mlb.fit_transform(df['equipements']), 
                         columns=[f'eq_{int(c)}' for c in mlb.classes_],
                         index=df.index)

# Fusion avec ton DataFrame principal
df = pd.concat([df, equip_ohe], axis=1)
création des colonnes de sécurité d'un csv pour validation
#df[['catu', 'catv', 'secu1', 'secu2', 'secu3', 'secu1_corr', 'secu2_corr', 'secu3_corr', 'equipements', 'eq_1','eq_2','eq_3','eq_4','eq_5','eq_6','eq_7','eq_8','eq_9']].to_csv("../data/processed/accidents_2019_2023_secu.csv", index=False)
#Suppression des colonnes inutiles
df.drop(columns=['secu1', 'secu2','secu3','secu1_corr','secu1_corr','secu1_corr', 'equipements'], inplace=True)